# 🔍 Explainability Demo

Demonstrates:
- SHAP value computation & plots
- Feature importance (intrinsic + permutation)
- Accent variation analysis
- Feature correlation

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import yaml
import matplotlib.pyplot as plt

from src.pipeline.data_generator import DataGenerator
from src.pipeline.data_loader import DataLoaderFactory
from src.models.classifier import SklearnClassifier, HybridClassifier
from src.training.trainer import Trainer
from src.explainability.shap_explainer import SHAPExplainer
from src.explainability.feature_importance import FeatureImportanceAnalyzer
from src.explainability.accent_analysis import AccentVariationAnalyzer

%matplotlib inline

In [ ]:
with open('../configs/default.yaml') as f:
    config = yaml.safe_load(f)

np.random.seed(config['project']['seed'])
DEMO_SAMPLES = 5000

## 1. Prepare Data & Train Model

In [ ]:
gen = DataGenerator(config)
X, y, accent_ids, noise_ids = gen.generate_fast(num_samples=DEMO_SAMPLES)
gen.save(X, y, accent_ids, noise_ids)

loader = DataLoaderFactory(config)
X_train, y_train, acc_train, _ = loader.load_arrays('train')
X_val, y_val, _, _ = loader.load_arrays('val')
X_test, y_test, acc_test, _ = loader.load_arrays('test')

# Train sklearn model (fast, SHAP-friendly)
model = SklearnClassifier(config)
trainer = Trainer(config)
trainer.train_sklearn(model, X_train, y_train, X_val, y_val)

print(f'Test samples: {len(y_test)}')

## 2. SHAP Analysis

In [ ]:
shap_explainer = SHAPExplainer(config)

# Build feature names
from src.features.extractor import FeatureExtractor
ext = FeatureExtractor(config['features'])
feature_names = [f'mfcc_{i}_mean' for i in range(ext.n_mfcc)]
feature_names += [f'feature_{i}' for i in range(len(feature_names), X_test.shape[1])]

# Compute SHAP values
shap_values = shap_explainer.explain(model, X_test[:200], feature_names)

# Single instance explanation
shap_explainer.explain_single(model, X_test[:100], X_test[0], feature_names)

print('SHAP analysis complete — check experiments/explainability/ for plots')

## 3. Feature Importance

In [ ]:
fi_analyzer = FeatureImportanceAnalyzer(config)

# Intrinsic (tree-based)
intrinsic = fi_analyzer.intrinsic_importance(model, feature_names)

# Permutation-based (takes a moment)
perm_imp = fi_analyzer.permutation_importance(
    model, X_test[:500], y_test[:500], feature_names, n_repeats=3
)

# Correlation
fi_analyzer.feature_correlation(X_test[:500], feature_names)

print('Feature importance analysis complete')

## 4. Accent Variation Analysis

In [ ]:
accent_analyzer = AccentVariationAnalyzer(config)
accent_results = accent_analyzer.analyze(model, X_test, y_test, acc_test)

# Feature distributions by accent
accent_analyzer.feature_distribution_by_accent(X_test, acc_test, feature_names)

# Display results
import pandas as pd
df = pd.DataFrame(accent_results).T
display(df.style.highlight_max(axis=0, color='lightgreen'))

## 5. Display Generated Plots

In [ ]:
from IPython.display import Image, display as ipy_display

plot_dir = '../experiments/explainability'
plots = ['shap_summary.png', 'shap_bar.png', 'accent_comparison.png',
         'accent_confusion_matrices.png', 'intrinsic_importance.png',
         'feature_correlation.png']

for p in plots:
    path = os.path.join(plot_dir, p)
    if os.path.exists(path):
        print(f'\n--- {p} ---')
        ipy_display(Image(filename=path, width=700))